In [1]:
import pandas as pd
import numpy as np
import networkx as nx
import matplotlib.pyplot as plt
import os
import seaborn
import pickle
from tqdm import tqdm
from pandas import HDFStore
from scipy.sparse.linalg import svds, eigs
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, f1_score, precision_score, recall_score

In [1]:
# Import necessary libraries
import pandas as pd

# Load the feature-engineered data from the HDF5 file
df_final_train = pd.read_hdf('storage_sample_stage4.h5', 'train_df', mode='r')
df_final_test = pd.read_hdf('storage_sample_stage4.h5', 'test_df', mode='r')

# Load the additional CSV files
train_after_eda = pd.read_csv('train_after_eda.csv')
train_y = pd.read_csv('train_y.csv')
test_after_eda = pd.read_csv('test_after_eda.csv')
test_y = pd.read_csv('test_y.csv')

# Combine the feature-engineered data with the additional data
df_final_train['indicator_link'] = train_y['indicator_link']  # Assuming 'indicator_link' is the column name in train_y
df_final_test['indicator_link'] = test_y['indicator_link']    # Assuming 'indicator_link' is the column name in test_y

df_combined_train = pd.concat([df_final_train, train_after_eda], axis=1)
df_combined_test = pd.concat([df_final_test, test_after_eda], axis=1)

# Save the final combined datasets to CSV files
df_combined_train.to_csv('final_combined_train_data.csv', index=False)
df_combined_test.to_csv('final_combined_test_data.csv', index=False)

print("Final combined train data saved as 'final_combined_train_data.csv'")
print("Final combined test data saved as 'final_combined_test_data.csv'")

Final combined train data saved as 'final_combined_train_data.csv'
Final combined test data saved as 'final_combined_test_data.csv'


In [ ]:
df_combined_train = pd.read_csv('final_combined_train_data.csv')
df_combined_train.head()
print(df_combined_train.value_counts('indicator_link'))

indicator_link
1.0    100002
Name: count, dtype: int64


In [ ]:
df_combined_train.head()
df_combined_train.tail()

In [ ]:
df_combined_test = pd.read_csv('final_combined_test_data.csv')
df_combined_test.head()
print(df_combined_test.value_counts('indicator_link'))

In [ ]:
df_combined_test.head()
df_combined_test.tail()

In [ ]:
df_combined_train.columns

In [ ]:
df_combined_train.tail()

In [ ]:
y_train = df_combined_train.indicator_link
y_test = df_combined_test.indicator_link

In [ ]:
df_combined_train.drop(['source_node', 'destination_node','indicator_link'],axis=1,inplace=True)
df_combined_test.drop(['source_node', 'destination_node','indicator_link'],axis=1,inplace=True)

In [ ]:
df_combined_train.tail()

In [ ]:
estimators = [10, 50, 100, 250, 450]
train_scores = []
test_scores = []

for i in estimators:
    clf = RandomForestClassifier(
        bootstrap=True,
        criterion='gini',
        max_depth=5,
        max_features='sqrt',
        min_samples_leaf=52,
        min_samples_split=120,
        n_estimators=i,
        n_jobs=-1,
        random_state=25
    )
    clf.fit(df_combined_train, y_train)
    train_sc = f1_score(y_train, clf.predict(df_combined_train))
    test_sc = f1_score(y_test, clf.predict(df_combined_test))
    train_scores.append(train_sc)
    test_scores.append(test_sc)
    print(f'Estimators = {i}, Train Score = {train_sc}, Test Score = {test_sc}')

plt.plot(estimators, train_scores, label='Train Score')
plt.plot(estimators, test_scores, label='Test Score')
plt.xlabel('Estimators')
plt.ylabel('F1 Score')
plt.title('Estimators vs F1 Score at Depth of 5')
plt.legend()
plt.show()

In [ ]:
depths = [3, 9, 11, 15, 20, 35, 50, 70, 130]
train_scores = []
test_scores = []

for i in depths:
    clf = RandomForestClassifier(
        bootstrap=True,
        criterion='gini',
        max_depth=i,
        max_features='sqrt',
        min_samples_leaf=52,
        min_samples_split=120,
        n_estimators=250,
        n_jobs=-1,
        random_state=25
    )
    clf.fit(df_combined_train, y_train)
    train_sc = f1_score(y_train, clf.predict(df_combined_train))
    test_sc = f1_score(y_test, clf.predict(df_combined_test))
    train_scores.append(train_sc)
    test_scores.append(test_sc)
    print(f'Depth = {i}, Train Score = {train_sc}, Test Score = {test_sc}')

plt.plot(depths, train_scores, label='Train Score')
plt.plot(depths, test_scores, label='Test Score')
plt.xlabel('Depth')
plt.ylabel('F1 Score')
plt.title('Depth vs F1 Score at Estimators = 250')
plt.legend()
plt.show()

In [ ]:
param_dist = {
    "n_estimators": sp_randint(105, 125),
    "max_depth": sp_randint(10, 15),
    "min_samples_split": sp_randint(110, 190),
    "min_samples_leaf": sp_randint(25, 65)
}

clf = RandomForestClassifier(random_state=25, n_jobs=-1)
rf_random = RandomizedSearchCV(
    clf,
    param_distributions=param_dist,
    n_iter=5,
    cv=5,  # Reduced from 10 to 5 for faster tuning
    scoring='f1',
    random_state=25,
    return_train_score=True
)

rf_random.fit(df_combined_train, y_train)
print('Mean Test Scores:', rf_random.cv_results_['mean_test_score'])
print('Mean Train Scores:', rf_random.cv_results_['mean_train_score'])
print('Best Estimator:', rf_random.best_estimator_)

In [ ]:
print(rf_random.best_estimator_)

In [ ]:
clf = RandomForestClassifier(
    bootstrap=True,
    criterion='gini',
    max_depth=14,
    max_features='sqrt',
    min_samples_leaf=28,
    min_samples_split=111,
    n_estimators=121,
    n_jobs=-1,
    random_state=25
)

clf.fit(df_combined_train, y_train)
y_train_pred = clf.predict(df_combined_train)
y_test_pred = clf.predict(df_combined_test)

print('Train F1 Score:', f1_score(y_train, y_train_pred))
print('Test F1 Score:', f1_score(y_test, y_test_pred))

In [ ]:
def plot_confusion_matrix(test_y, predict_y):
    C = confusion_matrix(test_y, predict_y)
    A = (((C.T) / (C.sum(axis=1))).T)
    B = (C / C.sum(axis=0))
    plt.figure(figsize=(20, 4))
    labels = [0, 1]
    cmap = sns.light_palette("blue")

    plt.subplot(1, 3, 1)
    sns.heatmap(C, annot=True, cmap=cmap, fmt=".3f", xticklabels=labels, yticklabels=labels)
    plt.xlabel('Predicted Class')
    plt.ylabel('Original Class')
    plt.title("Confusion Matrix")

    plt.subplot(1, 3, 2)
    sns.heatmap(B, annot=True, cmap=cmap, fmt=".3f", xticklabels=labels, yticklabels=labels)
    plt.xlabel('Predicted Class')
    plt.ylabel('Original Class')
    plt.title("Precision Matrix")

    plt.subplot(1, 3, 3)
    sns.heatmap(A, annot=True, cmap=cmap, fmt=".3f", xticklabels=labels, yticklabels=labels)
    plt.xlabel('Predicted Class')
    plt.ylabel('Original Class')
    plt.title("Recall Matrix")

    plt.show()

print('Train Confusion Matrix:')
plot_confusion_matrix(y_train, y_train_pred)

print('Test Confusion Matrix:')
plot_confusion_matrix(y_test, y_test_pred)

In [ ]:
fpr, tpr, thresholds = roc_curve(y_test, y_test_pred)
auc_score = auc(fpr, tpr)

plt.plot(fpr, tpr, color='navy', label=f'ROC curve (area = {auc_score:.2f})')
plt.xlabel('False Positive Rate')
plt.ylabel('True Positive Rate')
plt.title('Receiver Operating Characteristic with Test Data')
plt.legend()
plt.show()

In [ ]:
features = df_combined_train.columns
importances = clf.feature_importances_
indices = (np.argsort(importances))[-25:]
plt.figure(figsize=(10,12))
plt.title('Feature Importances')
plt.barh(range(len(indices)), importances[indices], color='r', align='center')
plt.yticks(range(len(indices)), [features[i] for i in indices])
plt.xlabel('Relative Importance')
plt.show()

In [ ]:
from sklearn.metrics import f1_score, confusion_matrix, roc_curve, auc
import seaborn as sns
import matplotlib.pyplot as plt

# Calculate F1 scores for train and test data
train_f1 = f1_score(y_train, y_train_pred)
test_f1 = f1_score(y_test, y_test_pred)

print(f'Train F1 Score: {train_f1}')
print(f'Test F1 Score: {test_f1}')

# Plot Confusion Matrix
def plot_confusion_matrix(test_y, predict_y):
    C = confusion_matrix(test_y, predict_y)
    A = (((C.T) / (C.sum(axis=1))).T)
    B = (C / C.sum(axis=0))
    plt.figure(figsize=(20, 4))
    labels = [0, 1]
    cmap = sns.light_palette("blue")

    plt.subplot(1, 3, 1)
    sns.heatmap(C, annot=True, cmap=cmap, fmt=".3f", xticklabels=labels, yticklabels=labels)
    plt.xlabel('Predicted Class')
    plt.ylabel('Original Class')
    plt.title("Confusion Matrix")

    plt.subplot(1, 3, 2)
    sns.heatmap(B, annot=True, cmap=cmap, fmt=".3f", xticklabels=labels, yticklabels=labels)
    plt.xlabel('Predicted Class')
    plt.ylabel('Original Class')
    plt.title("Precision Matrix")

    plt.subplot(1, 3, 3)
    sns.heatmap(A, annot=True, cmap=cmap, fmt=".3f", xticklabels=labels, yticklabels=labels)
    plt.xlabel('Predicted Class')
    plt.ylabel('Original Class')
    plt.title("Recall Matrix")

    plt.show()

print('Train Confusion Matrix:')
plot_confusion_matrix(y_train, y_train_pred)

print('Test Confusion Matrix:')
plot_confusion_matrix(y_test, y_test_pred)

# Plot ROC Curve
fpr, tpr, thresholds = roc_curve(y_test, y_test_pred)
auc_score = auc(fpr, tpr)

plt.plot(fpr, tpr, color='navy', label=f'ROC curve (area = {auc_score:.2f})')
plt.xlabel('False Positive Rate')
plt.ylabel('True Positive Rate')
plt.title('Receiver Operating Characteristic with Test Data')
plt.legend()
plt.show()

In [ ]:
# Function to recommend friends based on predictions
def recommend_friends(input_friend_id, df_test, clf):
    # Check if the input_friend_id exists in the dataframe
    if input_friend_id not in df_test.index:
        print(f"Friend ID {input_friend_id} not found in the test dataset.")
        return
    
    # Get the row corresponding to the input friend
    input_data = df_test[df_test.index == input_friend_id]
    
    # Predict probabilities for the input data
    probabilities = clf.predict_proba(input_data)[:, 1]
    
    # Sort and recommend friends based on probabilities
    recommendations = np.argsort(probabilities)[::-1][:5]  # Top 5 recommendations
    print(f"Recommended Friends for Friend ID {input_friend_id}: {recommendations}")
    print("Reason for Recommendation: High probability of forming a link based on features like common followers, followees, and graph metrics.")

# Example: Recommend friends for a specific friend ID
recommend_friends(12346, df_combined_test, clf)

In [ ]:
# Function to test the model performance for different source nodes
def test_model_performance(source_nodes, df_test, clf):
    for source_node in source_nodes:
        if source_node not in df_test.index:
            print(f"Source Node {source_node} not found in the test dataset.")
            continue
        
        # Get the row corresponding to the source node
        input_data = df_test[df_test.index == source_node]
        
        # Predict probabilities for the input data
        probabilities = clf.predict_proba(input_data)[:, 1]
        
        # Sort and recommend friends based on probabilities
        recommendations = np.argsort(probabilities)[::-1][:5]  # Top 5 recommendations
        print(f"Source Node: {source_node}")
        print(f"Recommended Friends: {recommendations}")
        print("Reason for Recommendation: High probability of forming a link based on features like common followers, followees, and graph metrics.")
        print("-" * 50)

# Example: Test the model with a few source nodes
source_nodes_to_test = [12345, 947487]  # Replace with valid source node IDs
test_model_performance(source_nodes_to_test, df_combined_test, clf)

In [ ]:
#  SVM - RBF KERNEL

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.svm import SVC
from sklearn.metrics import f1_score, accuracy_score, precision_score, recall_score, confusion_matrix, roc_curve, auc
from sklearn.preprocessing import StandardScaler

In [ ]:
# Load the combined train and test datasets
df_combined_train = pd.read_csv('final_combined_train_data.csv')
df_combined_test = pd.read_csv('final_combined_test_data.csv')

# Extract the target variable
y_train = df_combined_train['indicator_link']
y_test = df_combined_test['indicator_link']

# Drop unnecessary columns
X_train = df_combined_train.drop(['source_node', 'destination_node', 'indicator_link'], axis=1)
X_test = df_combined_test.drop(['source_node', 'destination_node', 'indicator_link'], axis=1)

In [ ]:
# Standardize the data (SVM is sensitive to feature scaling)
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

In [ ]:
# Train the SVM model with RBF kernel
svm_clf = SVC(kernel='rbf', probability=True, random_state=25)
svm_clf.fit(X_train_scaled, y_train)

# Predict on train and test data
y_train_pred = svm_clf.predict(X_train_scaled)
y_test_pred = svm_clf.predict(X_test_scaled)

In [ ]:
# Calculate evaluation metrics
train_f1 = f1_score(y_train, y_train_pred)
test_f1 = f1_score(y_test, y_test_pred)
train_accuracy = accuracy_score(y_train, y_train_pred)
test_accuracy = accuracy_score(y_test, y_test_pred)
train_precision = precision_score(y_train, y_train_pred)
test_precision = precision_score(y_test, y_test_pred)
train_recall = recall_score(y_train, y_train_pred)
test_recall = recall_score(y_test, y_test_pred)

# Print evaluation metrics
print(f"Train F1 Score: {train_f1}")
print(f"Test F1 Score: {test_f1}")
print(f"Train Accuracy: {train_accuracy}")
print(f"Test Accuracy: {test_accuracy}")
print(f"Train Precision: {train_precision}")
print(f"Test Precision: {test_precision}")
print(f"Train Recall: {train_recall}")
print(f"Test Recall: {test_recall}")

In [ ]:
def plot_confusion_matrix(test_y, predict_y):
    C = confusion_matrix(test_y, predict_y)
    A = (((C.T) / (C.sum(axis=1))).T)
    B = (C / C.sum(axis=0))
    plt.figure(figsize=(20, 4))
    labels = [0, 1]
    cmap = sns.light_palette("blue")

    plt.subplot(1, 3, 1)
    sns.heatmap(C, annot=True, cmap=cmap, fmt=".3f", xticklabels=labels, yticklabels=labels)
    plt.xlabel('Predicted Class')
    plt.ylabel('Original Class')
    plt.title("Confusion Matrix")

    plt.subplot(1, 3, 2)
    sns.heatmap(B, annot=True, cmap=cmap, fmt=".3f", xticklabels=labels, yticklabels=labels)
    plt.xlabel('Predicted Class')
    plt.ylabel('Original Class')
    plt.title("Precision Matrix")

    plt.subplot(1, 3, 3)
    sns.heatmap(A, annot=True, cmap=cmap, fmt=".3f", xticklabels=labels, yticklabels=labels)
    plt.xlabel('Predicted Class')
    plt.ylabel('Original Class')
    plt.title("Recall Matrix")

    plt.show()

print('Train Confusion Matrix:')
plot_confusion_matrix(y_train, y_train_pred)

print('Test Confusion Matrix:')
plot_confusion_matrix(y_test, y_test_pred)

In [ ]:
# Calculate probabilities for ROC curve
y_test_prob = svm_clf.predict_proba(X_test_scaled)[:, 1]

# Compute ROC curve and AUC
fpr, tpr, thresholds = roc_curve(y_test, y_test_prob)
auc_score = auc(fpr, tpr)

# Plot ROC curve
plt.plot(fpr, tpr, color='navy', label=f'ROC curve (area = {auc_score:.2f})')
plt.xlabel('False Positive Rate')
plt.ylabel('True Positive Rate')
plt.title('Receiver Operating Characteristic with Test Data')
plt.legend()
plt.show()

In [ ]:
# Function to recommend friends based on predictions
def recommend_friends(input_friend_id, df_test, clf, scaler):
    if input_friend_id not in df_test.index:
        print(f"Friend ID {input_friend_id} not found in the test dataset.")
        return
    
    input_data = df_test[df_test.index == input_friend_id]
    input_data_scaled = scaler.transform(input_data)
    probabilities = clf.predict_proba(input_data_scaled)[:, 1]
    recommendations = np.argsort(probabilities)[::-1][:5]
    print(f"Recommended Friends for Friend ID {input_friend_id}: {recommendations}")
    print("Reason for Recommendation: High probability of forming a link based on features like common followers, followees, and graph metrics.")

# Example: Recommend friends for a specific friend ID
recommend_friends(12346, df_combined_test, svm_clf, scaler)

In [ ]:
# Function to test the model performance for different source nodes
def test_model_performance(source_nodes, df_test, clf, scaler):
    for source_node in source_nodes:
        if source_node not in df_test.index:
            print(f"Source Node {source_node} not found in the test dataset.")
            continue
        
        input_data = df_test[df_test.index == source_node]
        input_data_scaled = scaler.transform(input_data)
        probabilities = clf.predict_proba(input_data_scaled)[:, 1]
        recommendations = np.argsort(probabilities)[::-1][:5]
        print(f"Source Node: {source_node}")
        print(f"Recommended Friends: {recommendations}")
        print("Reason for Recommendation: High probability of forming a link based on features like common followers, followees, and graph metrics.")
        print("-" * 50)

# Example: Test the model with a few source nodes
source_nodes_to_test = [12345, 947487]
test_model_performance(source_nodes_to_test, df_combined_test, svm_clf, scaler)

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from xgboost import XGBClassifier
from sklearn.metrics import f1_score, accuracy_score, precision_score, recall_score, confusion_matrix, roc_curve, auc
from sklearn.preprocessing import StandardScaler

In [ ]:
# Load the combined train and test datasets
df_combined_train = pd.read_csv('final_combined_train_data.csv')
df_combined_test = pd.read_csv('final_combined_test_data.csv')

# Extract the target variable
y_train = df_combined_train['indicator_link']
y_test = df_combined_test['indicator_link']

# Drop unnecessary columns
X_train = df_combined_train.drop(['source_node', 'destination_node', 'indicator_link'], axis=1)
X_test = df_combined_test.drop(['source_node', 'destination_node', 'indicator_link'], axis=1)

In [ ]:
# Standardize the data (XGBoost handles scaling internally, but scaling can still help)
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

In [ ]:
# Train the XGBoost model
xgb_clf = XGBClassifier(
    n_estimators=100,  # Number of trees
    max_depth=6,       # Maximum depth of a tree
    learning_rate=0.1, # Step size shrinkage
    subsample=0.8,     # Subsample ratio of the training instances
    colsample_bytree=0.8,  # Subsample ratio of columns when constructing each tree
    random_state=25,
    use_label_encoder=False,
    eval_metric='logloss'
)

xgb_clf.fit(X_train_scaled, y_train)

# Predict on train and test data
y_train_pred = xgb_clf.predict(X_train_scaled)
y_test_pred = xgb_clf.predict(X_test_scaled)

In [ ]:
# Calculate evaluation metrics
train_f1 = f1_score(y_train, y_train_pred)
test_f1 = f1_score(y_test, y_test_pred)
train_accuracy = accuracy_score(y_train, y_train_pred)
test_accuracy = accuracy_score(y_test, y_test_pred)
train_precision = precision_score(y_train, y_train_pred)
test_precision = precision_score(y_test, y_test_pred)
train_recall = recall_score(y_train, y_train_pred)
test_recall = recall_score(y_test, y_test_pred)

# Print evaluation metrics
print(f"Train F1 Score: {train_f1}")
print(f"Test F1 Score: {test_f1}")
print(f"Train Accuracy: {train_accuracy}")
print(f"Test Accuracy: {test_accuracy}")
print(f"Train Precision: {train_precision}")
print(f"Test Precision: {test_precision}")
print(f"Train Recall: {train_recall}")
print(f"Test Recall: {test_recall}")

In [ ]:
def plot_confusion_matrix(test_y, predict_y):
    C = confusion_matrix(test_y, predict_y)
    A = (((C.T) / (C.sum(axis=1))).T)
    B = (C / C.sum(axis=0))
    plt.figure(figsize=(20, 4))
    labels = [0, 1]
    cmap = sns.light_palette("blue")

    plt.subplot(1, 3, 1)
    sns.heatmap(C, annot=True, cmap=cmap, fmt=".3f", xticklabels=labels, yticklabels=labels)
    plt.xlabel('Predicted Class')
    plt.ylabel('Original Class')
    plt.title("Confusion Matrix")

    plt.subplot(1, 3, 2)
    sns.heatmap(B, annot=True, cmap=cmap, fmt=".3f", xticklabels=labels, yticklabels=labels)
    plt.xlabel('Predicted Class')
    plt.ylabel('Original Class')
    plt.title("Precision Matrix")

    plt.subplot(1, 3, 3)
    sns.heatmap(A, annot=True, cmap=cmap, fmt=".3f", xticklabels=labels, yticklabels=labels)
    plt.xlabel('Predicted Class')
    plt.ylabel('Original Class')
    plt.title("Recall Matrix")

    plt.show()

print('Train Confusion Matrix:')
plot_confusion_matrix(y_train, y_train_pred)

print('Test Confusion Matrix:')
plot_confusion_matrix(y_test, y_test_pred)

In [ ]:
# Calculate probabilities for ROC curve
y_test_prob = xgb_clf.predict_proba(X_test_scaled)[:, 1]

# Compute ROC curve and AUC
fpr, tpr, thresholds = roc_curve(y_test, y_test_prob)
auc_score = auc(fpr, tpr)

# Plot ROC curve
plt.plot(fpr, tpr, color='navy', label=f'ROC curve (area = {auc_score:.2f})')
plt.xlabel('False Positive Rate')
plt.ylabel('True Positive Rate')
plt.title('Receiver Operating Characteristic with Test Data')
plt.legend()
plt.show()

In [ ]:
# Function to recommend friends based on predictions
def recommend_friends(input_friend_id, df_test, clf, scaler):
    if input_friend_id not in df_test.index:
        print(f"Friend ID {input_friend_id} not found in the test dataset.")
        return
    
    input_data = df_test[df_test.index == input_friend_id]
    input_data_scaled = scaler.transform(input_data)
    probabilities = clf.predict_proba(input_data_scaled)[:, 1]
    recommendations = np.argsort(probabilities)[::-1][:5]
    print(f"Recommended Friends for Friend ID {input_friend_id}: {recommendations}")
    print("Reason for Recommendation: High probability of forming a link based on features like common followers, followees, and graph metrics.")

# Example: Recommend friends for a specific friend ID
recommend_friends(12346, df_combined_test, xgb_clf, scaler)

In [ ]:
# Function to test the model performance for different source nodes
def test_model_performance(source_nodes, df_test, clf, scaler):
    for source_node in source_nodes:
        if source_node not in df_test.index:
            print(f"Source Node {source_node} not found in the test dataset.")
            continue
        
        input_data = df_test[df_test.index == source_node]
        input_data_scaled = scaler.transform(input_data)
        probabilities = clf.predict_proba(input_data_scaled)[:, 1]
        recommendations = np.argsort(probabilities)[::-1][:5]
        print(f"Source Node: {source_node}")
        print(f"Recommended Friends: {recommendations}")
        print("Reason for Recommendation: High probability of forming a link based on features like common followers, followees, and graph metrics.")
        print("-" * 50)

# Example: Test the model with a few source nodes
source_nodes_to_test = [12345, 947487]
test_model_performance(source_nodes_to_test, df_combined_test, xgb_clf, scaler)

In [1]:
import pickle

# Assuming you have a pickle file named 'my_data.pkl'
# that contains a dictionary like {'name': 'Alice', 'age': 30}

try:
    with open('missing_edges_final.p', 'rb') as f:
        loaded_data = pickle.load(f)
        print("Loaded data:", loaded_data)
        print("Type of loaded data:", type(loaded_data))
except FileNotFoundError:
    print("Error: 'missing_edges_final.p' not found. Please ensure the file exists.")
except Exception as e:
    print(f"An error occurred: {e}")

Loaded data: <dict containing missing edges - output truncated due to 169MB size limit>
Type of loaded data: <class 'dict'>
